# Nav2 vs pi0.5 — Navigation Comparison Report

This notebook compares the **Nav2 (DWB) baseline** against the **pi0.5 VLA policy** using
**per-episode metric tables and spatial trajectory overlays**, instead of training-style
line/loss graphs. This choice is deliberate: Nav2 is a near-deterministic planner with a
high success rate, while pi0.5 is a learned policy with more variable behaviour — a single
aggregate curve would hide *where* and *how* the two diverge. Tables + trajectory overlays +
targeted failure case studies are more informative here.

## How the input data is produced

Run `eval_logger_node.py` (in `Mir250/mir_navigation/`) as a passive observer alongside
either controller — it only listens to `/episode_goal`, odometry and `cmd_vel`, so it does
not interfere with Nav2 or the pi0.5 inference node:

```bash
# While mir_random_nav.py (Nav2) is driving:
python3 eval_logger_node.py --ros-args -p controller_name:=nav2 -p map_name:=maze

# While goal_monitor_node.py + inference_ros2_node.py (pi0.5) are driving:
python3 eval_logger_node.py --ros-args -p controller_name:=pi05 -p map_name:=maze \
    -p cmd_vel_topic:=/diff_cont/cmd_vel_unstamped
```

Both runs append to the same `eval_metrics.csv` (default location `~/nav_eval_logs/`), with
per-episode trajectories saved under `~/nav_eval_logs/trajectories/`. Repeat for every map to
build up a comparison dataset.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import yaml
from PIL import Image

pd.set_option("display.width", 120)

# ---- Configuration: adjust to match where eval_logger_node.py wrote its output ----
OUTPUT_DIR = Path.home() / "nav_eval_logs"
EVAL_CSV = OUTPUT_DIR / "eval_metrics.csv"
TRAJ_DIR = OUTPUT_DIR / "trajectories"

# Repo maps (world_name.yaml + world_name.pgm), used for the trajectory overlay background
MAPS_DIR = Path("../../../Maps")  # relative to Mir250/mir_navigation/analysis/

print(f"EVAL_CSV : {EVAL_CSV} (exists={EVAL_CSV.exists()})")
print(f"TRAJ_DIR : {TRAJ_DIR} (exists={TRAJ_DIR.exists()})")
print(f"MAPS_DIR : {MAPS_DIR.resolve()} (exists={MAPS_DIR.exists()})")


In [ ]:
## Load per-episode metrics
if not EVAL_CSV.exists():
    print(
        f"No eval_metrics.csv found at {EVAL_CSV}.\n"
        "Run eval_logger_node.py for both 'nav2' and 'pi05' controllers first "
        "(see instructions in the first cell)."
    )
    df = pd.DataFrame()
else:
    df = pd.read_csv(EVAL_CSV)
    print(f"Loaded {len(df)} episodes from {EVAL_CSV}")

df.head()


In [ ]:
## Aggregated comparison table (controller x map)
if df.empty:
    summary = pd.DataFrame()
else:
    def _agg(g: pd.DataFrame) -> pd.Series:
        n = len(g)
        failed = g[g["result"] != "success"]
        return pd.Series({
            "n_episodes": n,
            "success_rate_%": 100.0 * (g["success"].sum() / n),
            "timeout_rate_%": 100.0 * ((g["result"] == "timeout").sum() / n),
            "time_s_mean": g["time_s"].mean(),
            "time_s_std": g["time_s"].std(),
            "path_length_m_mean": g["path_length_m"].mean(),
            "path_efficiency_mean": g["path_efficiency"].mean(),
            "final_dist_m_mean_on_failure": failed["final_dist_m"].mean() if len(failed) else np.nan,
            "mean_abs_ang_vel": g["mean_abs_ang_vel"].mean(),
            "ang_vel_std_mean": g["ang_vel_std"].mean(),
        })

    rows = []
    for (controller, map_name), g in df.groupby(["controller", "map"]):
        row = _agg(g)
        row["controller"] = controller
        row["map"] = map_name
        rows.append(row)
    summary = pd.DataFrame(rows)
    cols = ["controller", "map"] + [c for c in summary.columns if c not in ("controller", "map")]
    summary = summary[cols].round(3)

summary


**Reading the table:** `success_rate_%` and `timeout_rate_%` are expected to strongly favour
Nav2. The more interesting columns for characterising *how* pi0.5 differs are
`path_efficiency_mean` (straight-line distance / actual path driven — lower means more
wandering/indirect trajectories), `final_dist_m_mean_on_failure` (how close pi0.5 gets before
giving up, vs. wandering far off), and `ang_vel_std_mean` (higher means jerkier, less smooth
steering — common for learned policies vs. the optimized DWB sampler).


In [ ]:
## Map loading helper (for the trajectory overlay background)
def load_map(map_name: str):
    """Return (image_array, resolution, origin_xy) for a ROS map yaml+pgm pair."""
    yaml_path = MAPS_DIR / f"{map_name}.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError(f"Map yaml not found: {yaml_path.resolve()}")
    with open(yaml_path) as f:
        meta = yaml.safe_load(f)
    pgm_path = yaml_path.parent / meta["image"]
    img = np.array(Image.open(pgm_path).convert("L"))
    resolution = float(meta["resolution"])
    origin = meta["origin"]  # [x, y, yaw]
    return img, resolution, (float(origin[0]), float(origin[1]))


def world_to_pixel(x, y, img_h, resolution, origin_xy):
    """ROS map convention: pgm row 0 = top = map y-max."""
    ox, oy = origin_xy
    col = (x - ox) / resolution
    row = img_h - (y - oy) / resolution
    return col, row


In [ ]:
## Trajectory overlay plot: Nav2 vs pi0.5 on top of the map
COLORS = {"nav2": "tab:blue", "pi05": "tab:orange"}

def plot_trajectories(map_name: str, controllers=("nav2", "pi05")):
    img, resolution, origin_xy = load_map(map_name)
    img_h = img.shape[0]

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(img, cmap="gray", origin="upper")

    episodes = df[df["map"] == map_name]
    for controller in controllers:
        color = COLORS.get(controller, None)
        ep_rows = episodes[episodes["controller"] == controller]
        label_used = False
        for _, ep in ep_rows.iterrows():
            traj_path = TRAJ_DIR / ep["trajectory_file"]
            if not traj_path.exists():
                continue
            tdf = pd.read_csv(traj_path)
            px, py = world_to_pixel(tdf["x"].values, tdf["y"].values, img_h, resolution, origin_xy)
            style = "-" if ep["success"] == 1 else "--"
            ax.plot(px, py, style, color=color, linewidth=1.3, alpha=0.8,
                     label=controller if not label_used else None)
            label_used = True
            gx, gy = world_to_pixel(ep["goal_x"], ep["goal_y"], img_h, resolution, origin_xy)
            ax.plot(gx, gy, "*", color=color, markersize=10)
        sx, sy = world_to_pixel(0.0, 0.0, img_h, resolution, origin_xy)
    ax.plot(sx, sy, "ko", markersize=6, label="start")

    ax.set_title(f"Trajectories on '{map_name}' — solid = success, dashed = failure/timeout")
    ax.legend(loc="upper right")
    ax.set_xlabel("pixel (x)")
    ax.set_ylabel("pixel (y)")
    plt.tight_layout()
    plt.show()


In [ ]:
## Plot overlays for every map present in the metrics
if not df.empty:
    for map_name in sorted(df["map"].unique()):
        plot_trajectories(map_name)
else:
    print("No data loaded yet — run eval_logger_node.py first.")


## Qualitative case studies: pi0.5 failures

Aggregate numbers explain *how much* pi0.5 underperforms; they don't explain *why*. The cell
below surfaces the worst pi0.5 episodes (timed out, or finished furthest from the goal) so you
can go back to the corresponding RViz recording / screenshot and describe the failure mode
(e.g. stuck against an obstacle, drifted off in the wrong direction, ignored the prompt).


In [ ]:
if df.empty:
    print("No data loaded yet.")
else:
    failures = df[(df["controller"] == "pi05") & (df["success"] == 0)].copy()
    failures = failures.sort_values("final_dist_m", ascending=False)
    cols = ["episode_id", "map", "result", "time_s", "path_efficiency",
            "final_dist_m", "trajectory_file"]
    display(failures[cols].head(10))


## Summary

This workflow deliberately replaces training-style line/loss graphs with:

1. **Per-episode metric tables** (success rate, timeout rate, time-to-goal, path efficiency,
   command smoothness) grouped by controller and map.
2. **Spatial trajectory overlays** on the actual map, showing *where* pi0.5 diverges from the
   Nav2 baseline (solid = success, dashed = failure/timeout).
3. **Targeted qualitative case studies** of the worst pi0.5 episodes, pointing back to the
   original RViz recordings for manual failure-mode analysis.

This is more appropriate than aggregate curves given that Nav2 is a near-deterministic
planner with a high success rate, while pi0.5 is a learned policy whose behaviour is best
understood through concrete episodes rather than a single averaged trend.
